# Module 43 — Multi-Touch Attribution with Markov Chains

> **Track 4 · Marketing Analytics & Optimization**  
> **Difficulty**: Advanced  
> **Runtime**: ~10 minutes  
> **Key Libraries**: Pandas, NumPy, Plotly  
> **Prerequisite**: Module 41 (Marketing Mix Modeling)

## Table of Contents

1. [Business Context](#1-business-context)
2. [Setup & Imports](#2-setup--imports)
3. [Synthetic Customer Journeys](#3-synthetic-customer-journeys)
4. [Markov Chain Model](#4-markov-chain-model)
5. [Attribution Calculation](#5-attribution-calculation)
6. [Comparison with Other Methods](#6-comparison-with-other-methods)
7. [Visualization](#7-visualization)
8. [Key Business Takeaway](#8-key-business-takeaway)

---
## §1 · Business Context
    
### Why multi-touch attribution?

Multi-touch attribution **credits all touchpoints** in customer journey:
- **Fair allocation**: Credit each channel appropriately.
- **Budget optimization**: Invest in channels that drive conversions.
- **Journey understanding**: Understand how channels work together.

**Attribution methods**:
- **Last-touch**: All credit to last touchpoint.
- **First-touch**: All credit to first touchpoint.
- **Linear**: Equal credit to all touchpoints.
- **Markov**: Credit based on removal effect.

**Applications**:
1. **Channel valuation**: Understand true channel contribution.
2. **Budget allocation**: Invest in high-attribution channels.
3. **Campaign optimization**: Optimize touchpoint sequences.

In this notebook we will:
1. Generate synthetic customer journey data.
2. Build Markov chain transition model.
3. Calculate removal effect attribution.
4. Compare with last-touch and linear attribution.

---
## §2 · Setup & Imports

In [ ]:
# ── Reproducibility ──────────────────────────────────────────────
import numpy as np
import pandas as pd
import warnings

np.random.seed(42)
warnings.filterwarnings("ignore")

# ── Visualization ────────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px

# ── Display settings ─────────────────────────────────────────────
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
sns.set_theme(style="whitegrid")

print("✓ All imports loaded")

---
## §3 · Synthetic Customer Journeys

In [ ]:
# Generate synthetic customer journeys
N_JOURNEYS = 10000

# Marketing channels
channels = ['Paid Search', 'Social', 'Email', 'Display', 'Direct']

# Journey templates
journey_templates = [
    ['Paid Search', 'Conversion'],
    ['Social', 'Paid Search', 'Conversion'],
    ['Email', 'Conversion'],
    ['Display', 'Social', 'Paid Search', 'Conversion'],
    ['Paid Search', 'Email', 'Conversion'],
    ['Social', 'Conversion'],
    ['Display', 'Conversion'],
    ['Paid Search', 'Social', 'Email', 'Conversion'],
    ['Direct', 'Conversion'],
    ['Social', 'Display', 'Conversion'],
]

# Conversion rates by journey type
conversion_rates = [0.15, 0.12, 0.20, 0.08, 0.18, 0.10, 0.05, 0.14, 0.25, 0.09]

# Generate journeys
journeys = []
for i in range(N_JOURNEYS):
    template_idx = np.random.choice(len(journey_templates), p=[0.2, 0.15, 0.15, 0.1, 0.1, 0.1, 0.05, 0.05, 0.05, 0.05])
    journey = journey_templates[template_idx].copy()
    
    # Some journeys don't convert
    if np.random.random() > conversion_rates[template_idx]:
        journey[-1] = 'No Conversion'
    
    journeys.append({
        'journey_id': i,
        'touchpoints': journey,
        'converted': journey[-1] == 'Conversion'
    })

df_journeys = pd.DataFrame(journeys)

print(f"✓ Generated {N_JOURNEYS:,} customer journeys")
print(f"  Conversion rate: {df_journeys['converted'].mean()*100:.1f}%")
print(f"\nSample journeys:")
for i in range(3):
    print(f"  {df_journeys['touchpoints'].iloc[i]}")

---
## §4 · Markov Chain Model

In [ ]:
# Build transition matrix
all_states = channels + ['Conversion', 'No Conversion', 'Start']
n_states = len(all_states)
state_to_idx = {state: idx for idx, state in enumerate(all_states)}

# Initialize transition counts
transition_counts = np.zeros((n_states, n_states))

# Count transitions
for _, row in df_journeys.iterrows():
    journey = ['Start'] + row['touchpoints']
    for i in range(len(journey) - 1):
        from_state = state_to_idx[journey[i]]
        to_state = state_to_idx[journey[i+1]]
        transition_counts[from_state, to_state] += 1

# Normalize to get probabilities
row_sums = transition_counts.sum(axis=1, keepdims=True)
row_sums[row_sums == 0] = 1  # Avoid division by zero
transition_matrix = transition_counts / row_sums

print(f"✓ Transition matrix built")
print(f"  States: {all_states}")
print(f"  Matrix shape: {transition_matrix.shape}")
print(f"\nTransition probabilities from Start:")
start_idx = state_to_idx['Start']
for i, state in enumerate(all_states):
    if transition_matrix[start_idx, i] > 0:
        print(f"  Start → {state}: {transition_matrix[start_idx, i]:.4f}")

---
## §5 · Attribution Calculation

In [ ]:
# Calculate removal effect for each channel
def calculate_conversion_rate(transition_matrix, removed_channel=None):
    """Calculate conversion rate with optional channel removal."""
    # Simple simulation: start from 'Start', follow transitions
    n_simulations = 10000
    conversions = 0
    
    for _ in range(n_simulations):
        current_state = 'Start'
        
        for _ in range(10):  # Max 10 steps
            current_idx = state_to_idx[current_state]
            probabilities = transition_matrix[current_idx]
            
            # Remove channel if specified
            if removed_channel:
                removed_idx = state_to_idx[removed_channel]
                probabilities = probabilities.copy()
                probabilities[removed_idx] = 0
                # Redistribute probability
                if probabilities.sum() > 0:
                    probabilities = probabilities / probabilities.sum()
                else:
                    break
            
            # Choose next state
            next_state = np.random.choice(all_states, p=probabilities)
            
            if next_state in ['Conversion', 'No Conversion']:
                if next_state == 'Conversion':
                    conversions += 1
                break
            
            current_state = next_state
    
    return conversions / n_simulations

# Baseline conversion rate
baseline_rate = calculate_conversion_rate(transition_matrix)

# Calculate removal effect for each channel
attribution = {}
for channel in channels:
    rate_without = calculate_conversion_rate(transition_matrix, removed_channel=channel)
    attribution[channel] = baseline_rate - rate_without

# Normalize to 100%
total_attribution = sum(attribution.values())
for channel in attribution:
    attribution[channel] = attribution[channel] / total_attribution * 100

print(f"✓ Markov attribution calculated")
print(f"  Baseline conversion rate: {baseline_rate*100:.2f}%")
print(f"\nMarkov Attribution (%):")
for channel, attr in sorted(attribution.items(), key=lambda x: -x[1]):
    print(f"  {channel}: {attr:.2f}%")

---
## §6 · Comparison with Other Methods

In [ ]:
# Compare with last-touch and linear attribution
# Last-touch attribution
last_touch = {channel: 0 for channel in channels}
for _, row in df_journeys[df_journeys['converted']].iterrows():
    touchpoints = row['touchpoints']
    if len(touchpoints) > 1:
        last_channel = touchpoints[-2]  # Last channel before conversion
        if last_channel in last_touch:
            last_touch[last_channel] += 1

# Normalize
total_last_touch = sum(last_touch.values())
for channel in last_touch:
    last_touch[channel] = last_touch[channel] / total_last_touch * 100

# Linear attribution
linear = {channel: 0 for channel in channels}
for _, row in df_journeys[df_journeys['converted']].iterrows():
    touchpoints = [t for t in row['touchpoints'] if t in channels]
    for touchpoint in touchpoints:
        linear[touchpoint] += 1 / len(touchpoints)

# Normalize
total_linear = sum(linear.values())
for channel in linear:
    linear[channel] = linear[channel] / total_linear * 100

# Create comparison DataFrame
comparison = pd.DataFrame({
    'Channel': channels,
    'Last Touch': [last_touch[ch] for ch in channels],
    'Linear': [linear[ch] for ch in channels],
    'Markov': [attribution[ch] for ch in channels]
})

print(f"✓ Attribution comparison:")
print(comparison.to_string(index=False))

---
## §7 · Visualization

In [ ]:
# Visualize attribution comparison
fig = go.Figure()

fig.add_trace(go.Bar(
    x=comparison['Channel'],
    y=comparison['Last Touch'],
    name='Last Touch',
    marker_color='#636EFA'
))

fig.add_trace(go.Bar(
    x=comparison['Channel'],
    y=comparison['Linear'],
    name='Linear',
    marker_color='#EF553B'
))

fig.add_trace(go.Bar(
    x=comparison['Channel'],
    y=comparison['Markov'],
    name='Markov',
    marker_color='#00CC96'
))

fig.update_layout(
    title='Attribution Comparison by Method',
    xaxis_title='Channel',
    yaxis_title='Attribution (%)',
    barmode='group',
    height=400
)
fig.show()

print("💡 Key insights:")
print("   - Last-touch overvalues bottom-funnel channels")
print("   - Linear treats all touchpoints equally")
print("   - Markov captures true channel contribution")

---
## §8 · Key Business Takeaway

> ### 🎯 Action Items for Marketing Teams
>
> | Aspect | Recommendation |
> |--------|---------------|
> | **When to use** | Channel valuation, budget allocation, campaign planning |
> | **Method** | Markov chain attribution for accurate credit |
> | **Data** | Customer journey touchpoints with outcomes |
> | **Evaluation** | Compare with last-touch and linear methods |
> | **Deployment** | Monthly attribution refresh |
>
> ### 💡 Attribution Method Selection
>
> | Method | Pros | Cons | Best For |
> |--------|------|------|----------|
> | Last-touch | Simple | Overvalues bottom-funnel | Quick analysis |
> | Linear | Fair | Ignores importance | Equal contribution |
> | Markov | Accurate | Complex | True channel value |
>
> ### 🔑 Key Takeaways
>
> 1. **Markov attribution captures true channel contribution**.
> 2. **Removal effect** measures impact of removing a channel.
> 3. **Different methods give different results** - use Markov for accuracy.
> 4. **Customer journey data** is essential for multi-touch attribution.
> 5. **Regular attribution analysis** improves budget allocation.